# A1・A4・E3・E4：少数牌譜の試験抽出

モデル学習は行いません。既存20＋3特徴量・キャッシュは変更しません。判断直前の選択肢と分子・分母を監査するNotebookです。

P1〜P6を順に実行してください。既定は低3・高3の計6対局です。E4は共通現物とそれ以外の両方を持つ時だけ分母となります。A1は実打牌のテンパイ維持を要求しません。

## P1：Drive接続・固定コード取得・テスト

解析用の依存ライブラリは専用フォルダへ入れ、別プロセスで使います。Colab本体の読み込み済みnumpy等を置き換えません。ここでの合成テスト成功は、Drive実牌譜の全件検証成功を意味しません。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import sys, os, subprocess, json, urllib.request, hashlib
CODE_COMMIT = '80908486308f804fac97942923b7de597a715a6c'
SOURCE = Path('/tmp') / ('decision_feature_pilot_' + CODE_COMMIT[:12])
FILES = [
 'mahjong_research/__init__.py','mahjong_research/tenhou.py','mahjong_research/inputs.py',
 'mahjong_research/analysis.py','mahjong_research/legacy.py','choices/choices.py',
 'choices/real_rounds.xml','smoothing/study.py',
 'decision_features/__init__.py','decision_features/definitions.py',
 'decision_features/extract.py','decision_features/pipeline.py','decision_features/analysis.py',
 'decision_features/cli.py','decision_features/requirements.txt',
 'decision_features/tests/__init__.py','decision_features/tests/test_decisions.py',
 'decision_features/tests/fixtures/real_round6.xml']
for relative in FILES:
    target = SOURCE / relative
    target.parent.mkdir(parents=True, exist_ok=True)
    url = f'https://raw.githubusercontent.com/yuukienomoto/report_-syobon_n/{CODE_COMMIT}/research_v2/{relative}'
    content = urllib.request.urlopen(url).read()
    if target.exists() and target.read_bytes() != content:
        raise RuntimeError(f'固定コミットとローカルコードが不一致: {target}')
    if not target.exists(): target.write_bytes(content)
DEPS = SOURCE / '_dependencies'
stamp = DEPS / 'install_complete.txt'
requirements = SOURCE / 'decision_features/requirements.txt'
requirements_hash = hashlib.sha256(requirements.read_bytes()).hexdigest()
if not stamp.exists():
    subprocess.check_call([sys.executable,'-m','pip','install','-q','--upgrade','--target',str(DEPS),'-r',str(requirements)])
    stamp.write_text(requirements_hash)
elif stamp.read_text() != requirements_hash:
    raise RuntimeError('依存フォルダの版が不一致。新しいセッションでP1を実行してください。')
ENV = dict(os.environ, PYTHONPATH=os.pathsep.join(map(str,[DEPS,SOURCE,SOURCE/'choices',SOURCE/'smoothing'])), PYTHONIOENCODING='utf-8')
check = subprocess.run([sys.executable,'-m','unittest','decision_features.tests.test_decisions','-q'], env=ENV, text=True, capture_output=True)
print(check.stderr)
if check.returncode:
    print(check.stdout[-6000:])
    raise RuntimeError('テストが失敗しました。実牌譜抽出を開始せず出力を確認してください。')
print('固定コード:', CODE_COMMIT, '\nテスト成功。P2へ進んでください。')


## P2：入力と新しい出力先

RUN_DIRは既存のinput_plan.jsonとshared_games.csvがあるフォルダです。C5キャッシュのパスはinput_plan.jsonから取得します。ZIPを移動した場合だけZIP_OVERRIDEを設定してください。PLAYERはXML/C5の完全一致名（ここでは全角ｎ）です。

PER_GROUP=3は各段位期の前・中・後から選ぶ計6件の少数試験です。対象を明示する場合はEXPLICIT_IDSに元game_idを指定（最大60件）。初期1500と同じ時間帯、17段、共通対局は除外します。

In [ ]:
ROOT = Path('/content/drive/MyDrive/卒論')
RUN_DIR = ROOT / 'research_v2_runs/20261105T071722Z_5da6c9ad'
OUTPUT_ROOT = ROOT / 'research_v2_decision_features'
PLAYER = 'しょぼんｎ'
KEY = 'shobon'
ZIP_OVERRIDE = None  # 例: str(ROOT / '実在する牌譜.zip')
PER_GROUP = 3
EXPLICIT_IDS = None  # 例: ['2012052419gm-0029-0000-eb0d38e9', ...]
CONFIG = SOURCE / 'pilot_config.json'
PREVIEW = SOURCE / 'pilot_preview.json'
RESULT = SOURCE / 'pilot_result.json'
CONFIG.write_text(json.dumps(dict(run_dir=str(RUN_DIR), output_root=str(OUTPUT_ROOT),
    player=PLAYER, key=KEY, zip_override=ZIP_OVERRIDE, per_group=PER_GROUP,
    explicit_ids=EXPLICIT_IDS), ensure_ascii=False, indent=2), encoding='utf-8')
def run_step(action):
    subprocess.check_call([sys.executable,'-m','decision_features.cli',action,str(CONFIG),str(PREVIEW),str(RESULT)],env=ENV)
print('入力:', RUN_DIR, '\n新出力の親:', OUTPUT_ROOT)


## P3：入力監査・対象ID・処理件数の確認（抽出なし）

C5完了署名とハッシュ、プレイヤー、重複、対象集合を検査します。原ZIPの全対局解析は行いません。表示する対象ID・クラス・件数を確認してからP4へ進んでください。多人数リーチやカンの実際の網羅状況はP5で分かります。

In [ ]:
run_step('preview')


## P4：表示した少数対局だけを抽出・保存

元XMLのハッシュをC5と照合し、4項目を追加キャッシュへ保存します。この少数対局だけ旧20の分子・分母も再照合します。既存キャッシュを上書きしません。中断時はP1〜P4を再実行すると完了済みの成功キャッシュを再利用します。

P4は毎回新しい日時フォルダを作ります。抽出失敗は0埋めせず監査に残し、窓分析を停止します。

In [ ]:
run_step('extract')
OUT = Path(json.loads(RESULT.read_text(encoding='utf-8'))['output'])
print('今回の保存先:', OUT)


## P5：分子・分母、除外理由、失敗、観測機会を確認

成功した分母0はNaN。抽出失敗とは分けて確認します。exclusionsの理由は重複する場合があり、理由別件数の総和を打牌数とは解釈しません。既存の10/20窓を丸ごと抽出できていない場合は、窓の機会数・欠損率は未測定です。飛び飛びの試験対局を連結しません。

In [ ]:
import pandas as pd
from IPython.display import display, Markdown
for name in ['extraction_audit.csv','feature_opportunities.csv','exclusions.csv','window_coverage.csv','window_opportunities.csv']:
    print(name)
    display(pd.read_csv(OUT/name))
concentration = pd.read_csv(OUT/'round_concentration.csv')
print('機会が多い局（上位20行。全件はCSV）')
display(concentration.sort_values('denominator',ascending=False).head(20))
print('判断直前の手牌と公開河: ', OUT/'decision_traces.json')
print('局・イベント番号ごとの計数理由: ', OUT/'decision_events.csv')


## P6：日本語レポート・保存先の確認

少数試験の値は最終的な群間差ではありません。E3/E4等に実例がなければ「実牌譜正例は未検証」のままです。order_verified=False、full_history_confirmed=Falseを維持します。次は計数と牌姿を照合し、必要な局面の追加試験を検討してください。本Notebookは全件再解析・モデル学習を行いません。

In [ ]:
display(Markdown((OUT/'report_ja.md').read_text(encoding='utf-8')))
print('成果物:', OUT)
print('\n'.join(p.name for p in sorted(OUT.iterdir()) if p.is_file()))
